 # Converting a grid model from LTDS to pandapower


Pandapower with UK Power Networks

This tutorial shows some functionalities and studies that can be performed using the power flow capabilities of pandapower. It will demonstrate how to run power flow simulations in pandapower, how to analyse the grid and to investigate different use cases relying on the power flow engine of pandapower.

This tutorial has been created in collaboration with UK Power Networks (UKPN), the Distribution System Operator owning and operating the electricity network across London, the South East and the East of England.

The tutorial will use the real grids associated with the three licensed electricity distribution networks operated by UKPN (LPN, SPN and EPN). It will provide some examples of how pandapower can be used to run investigations and analyses using the open source data released by UKPN. UK Power Networks has provided this grid data as part of their LTDS CIM dataset release. It is a "Shared" dataset that requires special access. To request access:

·       Register and login to the UKPN Open Data Portal [ukpowernetworks.opendatasoft.com]

·       Visit the LTDS CIM [ukpowernetworks.opendatasoft.com] page and complete the Shared Data Request Form [ukpowernetworks.opendatasoft.com]



Once approved, CIM data is published as XML file attachments (one per licence area: EPN, SPN, LPN). You can download the XML files directly from the portal.

Import the pandapower library and the neccessary methods for the conversion as follows:

In [ ]:
import os
import pandapower as pp
from pandapower.converter.cim import from_cim as cim2pp
from pandapower.converter.cim.cim_classes import CimParser
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## LTDS to pandapower

First, we define the LTDS zip archive, which can be converted to pandapower. If there is no SSH profile available, there is an option to get some P and Q values from Excel. However, please take into account that there are some assumptions made, which limit the applicability of the data and make them unsuitable for all use cases.

Note: If you don't have the Excel files for demand and generation, you can get access to them via the following links:
demand:
https://ukpowernetworks.opendatasoft.com/explore/assets/ltds-table-3a-load-data-observed/view/
generation:
https://ukpowernetworks.opendatasoft.com/explore/assets/ltds-table-5-generation/view/

In [ ]:
# ltds_files is a list containing paths to files needed for the LTDS converter:
ltds_files = [r"path-to-ltds-zip"]
path_excel_demand = r"path-to-excel-input-files"
path_excel_generation = r"path-to-excel-input-files"
excel_column_name = 'Maximum_Demand_24_25_MW'
# the Excel data provides the maximum demand / generation. If you want to assume a specific loading,
# choose a scaling_factor between 0.1 and 1.0
scaling_factor = 1.0

cim_parser = CimParser(cgmes_version='ltds')
cim_parser.parse_files(ltds_files).prepare_cim_net().set_cim_data_types()
cim = cim_parser.cim

if os.path.isfile(path_excel_demand):
    excel_df_demand = pd.read_excel(path_excel_demand, sheet_name='Feuil1', skiprows=0)
else:
    excel_df_demand = pd.DataFrame()
if os.path.isfile(path_excel_generation):
    excel_df_generation = pd.read_excel(path_excel_generation, sheet_name='Feuil1', skiprows=0)
else:
    excel_df_generation = pd.DataFrame()
def format_uuid_no_dash(uuid_no_dash: str) -> str | None:
    if uuid_no_dash is None:
        return None
    s = str(uuid_no_dash).lstrip("_")
    if len(s) != 32:
        return uuid_no_dash
    parts = [s[0:8], s[8:12], s[12:16], s[16:20], s[20:32]]
    return '_' + '-'.join(parts)
# prepare the demand data
if not excel_df_demand.empty:
    if 'Season' in excel_df_demand:
        excel_df_demand = excel_df_demand.loc[excel_df_demand['Season'] == 'Winter']
    excel_df_demand = excel_df_demand.rename(columns={'Substation MRID': 'name_excel', excel_column_name: 'p_mw_excel'})
    if 'name_excel' not in excel_df_demand or 'p_mw_excel' not in excel_df_demand:
        # the Excel document is not valid
        excel_df_demand = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])
    excel_df_demand = excel_df_demand[['name_excel', 'p_mw_excel']]
    excel_df_demand['p_mw_excel'] = excel_df_demand['p_mw_excel'].astype(float)
    excel_df_demand = excel_df_demand.dropna(how='any')
    excel_df_demand = excel_df_demand.groupby('name_excel', as_index=False)['p_mw_excel'].sum()
else:
    excel_df_demand = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])
# prepare the generation data
if not excel_df_generation.empty:
    if 'Connected_Accepted' in excel_df_generation:
        excel_df_generation = excel_df_generation.loc[excel_df_generation['Connected_Accepted'] == 'Connected']
    excel_df_generation = excel_df_generation.rename(columns={'Substation MRID': 'name_excel', 'InstalledCapacity_MVA': 'p_mw_excel'})
    if 'name_excel' not in excel_df_generation or 'p_mw_excel' not in excel_df_generation:
        # the Excel document is not valid
        excel_df_generation = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])
    excel_df_generation['p_mw_excel'] = excel_df_generation['p_mw_excel'].astype(float)
    excel_df_generation = excel_df_generation.dropna(how='any')
    # note: there might be an issue with the UUID format, this will be fixed with the following line:
    excel_df_generation['name_excel'] = excel_df_generation['name_excel'].apply(format_uuid_no_dash)

    excel_df_generation = excel_df_generation.groupby('name_excel', as_index=False)['p_mw_excel'].sum()
else:
    excel_df_generation = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])

# add the loads to the SSH profile
cim['ssh']['EnergyConsumer'] = pd.concat([cim['ssh']['EnergyConsumer'], cim['eq']['EnergyConsumer'][['rdfId']]], ignore_index=True)
# get the Substation ID
sub = cim['eq']['Terminal'][['ConnectivityNode', 'ConductingEquipment']]
sub = sub.rename(columns={'ConnectivityNode': 'rdfId'})
sub = pd.merge(sub, cim['eq']['ConnectivityNode'][['rdfId', 'ConnectivityNodeContainer']], how='left', on='rdfId')
sub = sub.drop(columns=['rdfId']).drop_duplicates(subset=['ConductingEquipment'])
# adding substations to EnergyConsumer
cim['ssh']['EnergyConsumer']['sub'] = cim['ssh']['EnergyConsumer']['rdfId'].map(
    sub.set_index('ConductingEquipment')['ConnectivityNodeContainer'])
# identify duplications
cim['ssh']['EnergyConsumer']['dups'] = cim['ssh']['EnergyConsumer'].groupby('sub')['sub'].transform('count')
cim['ssh']['EnergyConsumer']['p'] = cim['ssh']['EnergyConsumer']['sub'].map(
    excel_df_demand.set_index('name_excel')['p_mw_excel']) / cim['ssh']['EnergyConsumer']['dups']
cim['ssh']['EnergyConsumer']['p'] = cim['ssh']['EnergyConsumer']['p'].fillna(0.) * scaling_factor
cim['ssh']['EnergyConsumer']['q'] = cim['ssh']['EnergyConsumer']['q'].fillna(0.)
cim['ssh']['EnergyConsumer']['inService'] = cim['ssh']['EnergyConsumer']['inService'].fillna(True)
cim['ssh']['EnergyConsumer'] = cim['ssh']['EnergyConsumer'].drop(columns=['sub', 'dups'])

# add the generation to the SSH profile
for one_asset in ['SynchronousMachine', 'PowerElectronicsConnection']:
    cim['ssh'][one_asset] = pd.concat([cim['ssh'][one_asset], cim['eq'][one_asset][['rdfId']]], ignore_index=True)
    # adding substations to generators
    cim['ssh'][one_asset]['sub'] = cim['ssh'][one_asset]['rdfId'].map(sub.set_index('ConductingEquipment')['ConnectivityNodeContainer'])
    # identify duplications
    cim['ssh'][one_asset]['dups'] = cim['ssh'][one_asset].groupby('sub')['sub'].transform('count')
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['sub'].map(
        excel_df_generation.set_index('name_excel')['p_mw_excel']) / cim['ssh'][one_asset]['dups']
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['p'].fillna(0.) * scaling_factor
    cim['ssh'][one_asset]['q'] = cim['ssh'][one_asset]['q'].fillna(0.)
    cim['ssh'][one_asset]['inService'] = cim['ssh'][one_asset]['inService'].fillna(True)
    cim['ssh'][one_asset] = cim['ssh'][one_asset].drop(columns=['sub', 'dups'])

for one_sw in ['Breaker', 'Disconnector', 'Switch', 'LoadBreakSwitch']:
    cim['ssh'][one_sw] = pd.concat([cim['ssh'][one_sw], cim['eq'][one_sw][['rdfId', 'normalOpen']].rename(columns={'normalOpen': 'open'})], ignore_index=True)
    cim['ssh'][one_sw]['inService'] = True

for one_asset in ['ExternalNetworkInjection', 'ConformLoad', 'NonConformLoad', 'StationSupply',
                  'AsynchronousMachine', 'EquivalentInjection']:
    cim['ssh'][one_asset] = pd.concat([cim['ssh'][one_asset], cim['eq'][one_asset][['rdfId']]], ignore_index=True)
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['p'].fillna(0.) * scaling_factor
    cim['ssh'][one_asset]['q'] = cim['ssh'][one_asset]['q'].fillna(0.)
    cim['ssh'][one_asset]['inService'] = cim['ssh'][one_asset]['inService'].fillna(True)

cim['ssh']['ExternalNetworkInjection']['referencePriority'] = cim['ssh']['ExternalNetworkInjection']['referencePriority'].fillna(1)
cim['ssh']['ExternalNetworkInjection']['controlEnabled'] = cim['ssh']['ExternalNetworkInjection']['controlEnabled'].fillna(True)
cim['ssh']['SynchronousMachine']['referencePriority'] = cim['ssh']['SynchronousMachine']['referencePriority'].fillna(0)
cim['ssh']['SynchronousMachine']['controlEnabled'] = cim['ssh']['SynchronousMachine']['controlEnabled'].fillna(False)
cim['ssh']['EquivalentInjection']['regulationStatus'] = cim['ssh']['EquivalentInjection']['regulationStatus'].fillna(False)

cim['ssh']['EnergySource'] = pd.concat([cim['ssh']['EnergySource'], cim['eq']['EnergySource'][['rdfId']]], ignore_index=True)
cim['ssh']['EnergySource']['activePower'] = cim['ssh']['EnergySource']['activePower'].fillna(0.)
cim['ssh']['EnergySource']['reactivePower'] = cim['ssh']['EnergySource']['reactivePower'].fillna(0.)
cim['ssh']['EnergySource']['inService'] = cim['ssh']['EnergySource']['inService'].fillna(True)

cim['ssh']['StaticVarCompensator'] = pd.concat([cim['ssh']['StaticVarCompensator'], cim['eq']['StaticVarCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['StaticVarCompensator']['q'] = cim['ssh']['StaticVarCompensator']['q'].fillna(0.)
cim['ssh']['StaticVarCompensator']['inService'] = cim['ssh']['StaticVarCompensator']['inService'].fillna(True)

# add the terminals
cim['ssh']['Terminal'] = pd.concat([cim['ssh']['Terminal'], cim['eq']['Terminal'][['rdfId']]], ignore_index=True)
cim['ssh']['Terminal']['connected'] = cim['ssh']['Terminal']['connected'].fillna(True)
# add the shunts
cim['ssh']['LinearShuntCompensator'] = pd.concat([cim['ssh']['LinearShuntCompensator'], cim['eq']['LinearShuntCompensator'][['rdfId', 'normalSections']].rename(
    columns={'normalSections': 'sections'})], ignore_index=True)
cim['ssh']['LinearShuntCompensator']['controlEnabled'] = cim['ssh']['LinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['LinearShuntCompensator']['inService'] = cim['ssh']['LinearShuntCompensator']['inService'].fillna(True)
cim['ssh']['NonlinearShuntCompensator'] = pd.concat([cim['ssh']['NonlinearShuntCompensator'], cim['eq']['NonlinearShuntCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['NonlinearShuntCompensator']['sections'] = cim['ssh']['NonlinearShuntCompensator']['sections'].fillna(0)
cim['ssh']['NonlinearShuntCompensator']['controlEnabled'] = cim['ssh']['NonlinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['NonlinearShuntCompensator']['inService'] = cim['ssh']['NonlinearShuntCompensator']['inService'].fillna(True)

# add the tap changer steps
cim['ssh']['RatioTapChanger'] = pd.concat([cim['ssh']['RatioTapChanger'], cim['eq']['RatioTapChanger'][['rdfId', 'neutralStep']].rename(
    columns={'neutralStep': 'step'})], ignore_index=True)
cim['ssh']['RatioTapChanger']['controlEnabled'] = cim['ssh']['RatioTapChanger']['controlEnabled'].fillna(False)
# add the TapChangerControls
cim['ssh']['TapChangerControl'] = pd.concat([cim['ssh']['TapChangerControl'], cim['eq']['TapChangerControl'][['rdfId']]], ignore_index=True)
cim['ssh']['TapChangerControl']['discrete'] = cim['ssh']['TapChangerControl']['discrete'].fillna(False)
cim['ssh']['TapChangerControl']['enabled'] = cim['ssh']['TapChangerControl']['enabled'].fillna(False)
cim['eq']['PowerTransformer']['inService'] = True

cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['EquivalentBranch'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['ACLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['DCLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)

# use the from_cim_dict to put in the modified CimParser
net = cim2pp.from_cim_dict(cim_parser=cim_parser, cim_version='LTDS', create_tap_controller=False)

# if there is no slack in the grid, create one
if net.gen.empty and net.ext_grid.empty and not net.sgen.empty:
    slack = net.sgen.loc[net.sgen.in_service].loc[net.sgen.p_mw == net.sgen.p_mw.max()]
    net.sgen = net.sgen.drop(slack.index[0])
    pp.create_gen(net, bus=slack.bus.iloc[0], p_mw=slack.p_mw.iloc[0], slack=True, in_service=True)

print('Conversion successful')

## Get an overview over your grid
Once the network is converted to pandapower, the data can be displayed:

In [ ]:
print(net)

## Export your grid
There are different options to export, for example as JSON, Excel or CSV:

In [ ]:
path_to_json = r'path-to-json.json'
path_to_excel = r'path-to-excel.xlsx'
path_to_csv = r'path-to-csv'
pp.to_json(net, path_to_json)
pp.to_excel(net, path_to_excel)
if os.path.isdir(path_to_csv):
    # for CSV, there is no pandapower method available
    for one_type in ['bus', 'line', 'impedance', 'trafo', 'trafo3w', 'load', 'sgen', 'gen']:
        net[one_type].to_csv(path_to_csv+'\\'+one_type+'.csv', sep=',')
else:
    print("Please provide a valid path for exporting the CSV data.")

## Display the nodes

In [ ]:
print(f"Overview over the nodes: {net.bus.describe()}")
print(f"The nodes: {net.bus.loc[:100]}")

## Display the lines

In [ ]:
print(f"Overview over the lines: {net.line.describe()}")
print(f"The lines: {net.line.loc[:100]}")

## Display the transformers

In [ ]:
# two winding transformers
print(f"Overview over the 2W transformers: {net.trafo.describe()}")
print(f"The 2W transformers: {net.trafo.loc[:100]}")
# three winding transformers
print(f"Overview over the 3W transformers: {net.trafo3w.describe()}")
print(f"The 3W transformers: {net.trafo3w.loc[:100]}")

## Display the loads

In [ ]:
print(f"Overview over the loads: {net.load.describe()}")
print(f"The loads: {net.load.loc[:100]}")

## Display the generation

In [ ]:
print(f"Overview over the PQ generators: {net.sgen.describe()}")
print(f"The PQ generators: {net.sgen.loc[:100]}")

print(f"Overview over the PV generators: {net.gen.describe()}")
print(f"The PV generators: {net.gen.loc[:100]}")

## Get only HV elements from the grid
In pandapower we are using pandas DataFrames, you can create your queries like you wish. Here is an example to get HV (110kV) elements from your grid.

In [ ]:
print("the HV nodes first")
print(net.bus.loc[(net.bus.vn_kv > 100) & (net.bus.vn_kv < 150)])

In [ ]:
print("now the HV lines")
net.line['vn_kv_bus'] = net.line.from_bus.map(net.bus.vn_kv)
print(net.line.loc[(net.line.vn_kv_bus > 100) & (net.line.vn_kv_bus < 150)])

In [ ]:
print("now the HV 2W trafos")
print(net.trafo.loc[(net.trafo.vn_hv_kv > 100) & (net.trafo.vn_hv_kv < 150)])

In [ ]:
print("now the HV 3W trafos")
print(net.trafo3w.loc[(net.trafo3w.vn_hv_kv > 100) & (net.trafo3w.vn_hv_kv < 150)])

In [ ]:
print("now the loads")
net.load['vn_kv_bus'] = net.load.bus.map(net.bus.vn_kv)
print(net.load.loc[(net.load.vn_kv_bus > 100) & (net.load.vn_kv_bus < 150)])

In [ ]:
print("now the PQ generators")
net.sgen['vn_kv_bus'] = net.sgen.bus.map(net.bus.vn_kv)
print(net.sgen.loc[(net.sgen.vn_kv_bus > 100) & (net.sgen.vn_kv_bus < 150)])

In [ ]:
print("now the PV generators")
net.gen['vn_kv_bus'] = net.gen.bus.map(net.bus.vn_kv)
print(net.gen.loc[(net.gen.vn_kv_bus > 100) & (net.gen.vn_kv_bus < 150)])